# Statistical Analysis of Laptop Data

This notebook will load the `cleaned-laptops-data.csv` dataset and perform several hypothesis tests to answer relevant analysis questions.

**Tests we will perform:**

1.  **Two-Sample T-test:** Is there a significant difference in the average price between HP and DELL laptops?
2.  **Chi-Square Test of Independence:** Is there an association between a laptop's brand and its RAM size?
3.  **Two-Proportion Z-test:** Do HP laptops have a significantly different proportion of "Windows 11 Pro" machines compared to DELL?
4.  **One-Sample T-test:** Is the average rating of all laptops significantly different from a "good" rating of 4.2?
5.  **Pearson Correlation Test:** Is there a statistically significant linear relationship between a laptop's `Price` and its `Rating`?

In [4]:
import pandas as pd
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest
import numpy as np


df = pd.read_csv('Cleaned_Laptops_Data.csv')
df.head()

,Brand_name,Device_name,Operating system,Ram(in GB),Storage(in GB),Price(in ₹),Discount(in percentages),Rating,Warrenty
0,Samsung,Samsung Galaxy Book4 Metal Intel Core i5 13th ...,Windows 11 Home,8,512,52500,32.0,4.4,1 Year Manufacturer Warranty for Laptop and 6 ...
1,CHUWI,CHUWI Intel Core i3 10th Gen 10100Y,Windows 11 Home,8,256,19990,50.0,4.1,1 Year Onsite Warranty
2,Acer,Acer Aspire 3 (2025) Intel Core i5 13th Gen 1334U,Windows 11 Home,16,512,41800,50.0,4.2,3 Years Warranty
3,Acer,Acer Aspire 3 Intel Celeron Dual Core,Windows 11 Home,8,128,15990,51.0,3.8,1 Year Warranty
4,Acer,Acer Aspire 3 Intel Core i3 12th Gen 1215U,Windows 11 Home,8,512,27990,34.0,4.2,1 Year Domestic Warranty


## Test 1: Two-Sample Independent T-test (HP vs. DELL Prices)

**Question:** Is there a significant difference in the average price between HP and DELL laptops?

* **Significance Level ($\alpha$):** 0.05
* **Null Hypothesis ($H_0$):** The mean price of HP laptops is equal to the mean price of DELL laptops.
    * $H_0: \mu_{HP} = \mu_{DELL}$
* **Alternative Hypothesis ($H_a$):** The mean price of HP laptops is *not* equal to the mean price of DELL laptops.
    * $H_a: \mu_{HP} \neq \mu_{DELL}$

We will use a two-sample t-test because we are comparing the means of two independent groups.

In [5]:
# 1. Prepare the data
hp_prices = df[df['Brand_name'] == 'HP']['Price(in ₹)'].dropna()
dell_prices = df[df['Brand_name'] == 'DELL']['Price(in ₹)'].dropna()

print(f"Average HP Price: ₹{hp_prices.mean():.2f}")
print(f"Average DELL Price: ₹{dell_prices.mean():.2f}")

t_statistic, p_value = stats.ttest_ind(hp_prices, dell_prices, equal_var=False)

print(f"\n--- T-test Results ---")
print(f"T-statistic: {t_statistic:.4f}")
print(f"P-value: {p_value:.4f}")

alpha = 0.05
if p_value < alpha:
    print(f"\nConclusion: Since the p-value ({p_value:.4f}) is less than {alpha}, we reject the null hypothesis.")
    print("There is a statistically significant difference in the average prices between HP and DELL laptops.")
else:
    print(f"\nConclusion: Since the p-value ({p_value:.4f}) is greater than {alpha}, we fail to reject the null hypothesis.")
    print("There is no statistically significant difference in the average prices between HP and DELL laptops.")

Average HP Price: ₹45027.68
Average DELL Price: ₹46523.20

--- T-test Results ---
T-statistic: -0.7093
P-value: 0.4792

Conclusion: Since the p-value (0.4792) is greater than 0.05, we fail to reject the null hypothesis.
There is no statistically significant difference in the average prices between HP and DELL laptops.


## Test 2: Chi-Square Test of Independence (Brand vs. RAM)

**Question:** Is there a significant association between a laptop's brand and the amount of RAM it has?

* **Significance Level ($\alpha$):** 0.05
* **Null Hypothesis ($H_0$):** `Brand_name` and `Ram(in GB)` are independent. (There is no association).
* **Alternative Hypothesis ($H_a$):** `Brand_name` and `Ram(in GB)` are dependent. (There is an association).

We use a Chi-Square test because we are checking for independence between two categorical variables.

In [7]:
# 1. Create a contingency table (crosstab)
common_brands = ['HP', 'DELL', 'Acer', 'ASUS', 'Lenovo', 'Samsung']
df_common_brands = df[df['Brand_name'].isin(common_brands)]

contingency_table = pd.crosstab(df_common_brands['Brand_name'], df_common_brands['Ram(in GB)'])

print("--- Contingency Table (Brand vs. RAM) ---")
print(contingency_table)

chi2_stat, p_value, dof, expected_freq = stats.chi2_contingency(contingency_table)

print(f"\n--- Chi-Square Test Results ---")
print(f"Chi2-statistic: {chi2_stat:.4f}")
print(f"P-value: {p_value:.4f}")
print(f"Degrees of Freedom: {dof}")

alpha = 0.05
if p_value < alpha:
    print(f"\nConclusion: Since the p-value ({p_value:.4f}) is less than {alpha}, we reject the null hypothesis.")
    print("There is a statistically significant association between laptop brand and the amount of RAM.")
else:
    print(f"\nConclusion: Since the p-value ({p_value:.4f}) is greater than {alpha}, we fail to reject the null hypothesis.")
    print("There is no statistically significant association between laptop brand and the amount of RAM.")

--- Contingency Table (Brand vs. RAM) ---
Ram(in GB)  4    8   12  16  24  32
Brand_name                         
ASUS         2   50   0  43   0   2
Acer         0   27   1  38   1   0
DELL         0   32   0  29   0   3
HP           2   74   1  46   0   4
Lenovo       3   49   1  49   7   0
Samsung      0  121   0  50   0   0

--- Chi-Square Test Results ---
Chi2-statistic: 76.9767
P-value: 0.0000
Degrees of Freedom: 25

Conclusion: Since the p-value (0.0000) is less than 0.05, we reject the null hypothesis.
There is a statistically significant association between laptop brand and the amount of RAM.


## Test 3: Two-Proportion Z-test (HP vs. DELL on 'Windows 11 Pro')

**Question:** Is the proportion of laptops with 'Windows 11 Pro' different for HP compared to DELL?

* **Significance Level ($\alpha$):** 0.05
* **Null Hypothesis ($H_0$):** The proportion of 'Windows 11 Pro' laptops is the same for HP and DELL.
    * $H_0: p_{HP} = p_{DELL}$
* **Alternative Hypothesis ($H_a$):** The proportion of 'Windows 11 Pro' laptops is *not* the same for HP and DELL.
    * $H_a: p_{HP} \neq p_{DELL}$

We use a two-proportion z-test to compare proportions from two independent groups.

In [8]:
# 1. Get the counts
n_hp = df[df['Brand_name'] == 'HP'].shape[0]
n_dell = df[df['Brand_name'] == 'DELL'].shape[0]

pro_hp = df[(df['Brand_name'] == 'HP') & (df['Operating system'] == 'Windows 11 Pro')].shape[0]
pro_dell = df[(df['Brand_name'] == 'DELL') & (df['Operating system'] == 'Windows 11 Pro')].shape[0]

print(f"HP: {pro_hp} 'Pro' laptops out of {n_hp} total. ({(pro_hp/n_hp)*100:.2f}%)")
print(f"DELL: {pro_dell} 'Pro' laptops out of {n_dell} total. ({(pro_dell/n_dell)*100:.2f}%)")

counts = np.array([pro_hp, pro_dell])
nobs = np.array([n_hp, n_dell])

z_stat, p_value = proportions_ztest(counts, nobs)

print(f"\n--- Two-Proportion Z-test Results ---")
print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.4f}")

alpha = 0.05
if p_value < alpha:
    print(f"\nConclusion: Since the p-value ({p_value:.4f}) is less than {alpha}, we reject the null hypothesis.")
    print("There is a statistically significant difference in the proportion of 'Windows 11 Pro' laptops between HP and DELL.")
else:
    print(f"\nConclusion: Since the p-value ({p_value:.4f}) is greater than {alpha}, we fail to reject the null hypothesis.")
    print("There is no statistically significant difference in the proportion of 'Windows 11 Pro' laptops between HP and DELL.")

HP: 13 'Pro' laptops out of 127 total. (10.24%)
DELL: 8 'Pro' laptops out of 64 total. (12.50%)

--- Two-Proportion Z-test Results ---
Z-statistic: -0.4721
P-value: 0.6369

Conclusion: Since the p-value (0.6369) is greater than 0.05, we fail to reject the null hypothesis.
There is no statistically significant difference in the proportion of 'Windows 11 Pro' laptops between HP and DELL.


## Test 4: One-Sample T-test (Average Rating)

**Question:** Is the mean rating of all laptops in this dataset significantly different from 4.2?

* **Significance Level ($\alpha$):** 0.05
* **Hypothetical Mean ($\mu_0$):** 4.2
* **Null Hypothesis ($H_0$):** The true mean rating of all laptops is 4.2.
    * $H_0: \mu = 4.2$
* **Alternative Hypothesis ($H_a$):** The true mean rating of all laptops is *not* 4.2.
    * $H_a: \mu \neq 4.2$

We use a one-sample t-test to compare our sample mean to a known or hypothesized population value.

In [9]:
# 1. Prepare the data
ratings = df['Rating'].dropna()
pop_mean = 4.2

print(f"Sample Mean Rating: {ratings.mean():.3f}")

t_statistic, p_value = stats.ttest_1samp(ratings, pop_mean)

print(f"\n--- One-Sample T-test Results ---")
print(f"T-statistic: {t_statistic:.4f}")
print(f"P-value: {p_value:.4f}")

alpha = 0.05
if p_value < alpha:
    print(f"\nConclusion: Since the p-value ({p_value:.4f}) is less than {alpha}, we reject the null hypothesis.")
    print(f"The sample mean rating ({ratings.mean():.3f}) is statistically significantly different from {pop_mean}.")
else:
    print(f"\nConclusion: Since the p-value ({p_value:.4f}) is greater than {alpha}, we fail to reject the null hypothesis.")
    print(f"There is no statistically significant difference between the sample mean rating ({ratings.mean():.3f}) and {pop_mean}.")

Sample Mean Rating: 4.115

--- One-Sample T-test Results ---
T-statistic: -3.6337
P-value: 0.0003

Conclusion: Since the p-value (0.0003) is less than 0.05, we reject the null hypothesis.
The sample mean rating (4.115) is statistically significantly different from 4.2.


## Test 5: Pearson Correlation Test (Price vs. Rating)

**Question:** Is there a statistically significant linear relationship between a laptop's price and its rating?

* **Significance Level ($\alpha$):** 0.05
* **Null Hypothesis ($H_0$):** There is no linear correlation between `Price(in ₹)` and `Rating`.
    * $H_0: \rho = 0$
* **Alternative Hypothesis ($H_a$):** There is a linear correlation between `Price(in ₹)` and `Rating`.
    * $H_a: \rho \neq 0$

We use a Pearson correlation test, which measures the strength and direction of a linear relationship.

In [10]:
# 1. Prepare the data (must drop NaNs from *both* columns)
df_corr = df[['Price(in ₹)', 'Rating']].dropna()

price = df_corr['Price(in ₹)']
rating = df_corr['Rating']

corr_coeff, p_value = stats.pearsonr(price, rating)

print(f"\n--- Pearson Correlation Test Results ---")
print(f"Pearson Correlation Coefficient: {corr_coeff:.4f}")
print(f"P-value: {p_value:.4f}")

alpha = 0.05
if p_value < alpha:
    print(f"\nConclusion: Since the p-value ({p_value:.4f}) is less than {alpha}, we reject the null hypothesis.")
    print(f"There is a statistically significant linear correlation between Price and Rating.")
    if corr_coeff > 0:
        print("The relationship is positive (as one increases, the other tends to increase).")
    else:
        print("The relationship is negative (as one increases, the other tends to decrease).")
else:
    print(f"\nConclusion: Since the p-value ({p_value:.4f}) is greater than {alpha}, we fail to reject the null hypothesis.")
    print("There is no statistically significant linear correlation between Price and Rating.")


--- Pearson Correlation Test Results ---
Pearson Correlation Coefficient: 0.1296
P-value: 0.0003

Conclusion: Since the p-value (0.0003) is less than 0.05, we reject the null hypothesis.
There is a statistically significant linear correlation between Price and Rating.
The relationship is positive (as one increases, the other tends to increase).
